# Clase 7 · Una red convolucional en vivo sobre un video real de la sala de clasificación
## Curso: Inteligencia Artificial Aplicada a la Producción Pesquera
### UTN FRCh · PesquerosEnIA · 10/11/2026

**Autor:** Ariel Giamportone | PesquerosEnIA

---

### Objetivo
Ver **con nuestros propios ojos** qué hace una red neuronal convolucional (CNN) sobre un video **real** de procesamiento a bordo: qué es una convolución, qué ve la red por dentro, por qué un modelo general **no sabe de pescado** y cómo en unos minutos le **enseñamos a ver peces** para aplicarlo cuadro por cuadro.

### El recorrido (≈ 20 minutos con GPU)
| Parte | Qué hacemos | Herramienta |
|---|---|---|
| 1 | Bajamos y miramos el video real | OpenCV |
| 2 | Hacemos una convolución **a mano** | OpenCV |
| 3 | Miramos **dentro** de una CNN entrenada | PyTorch |
| 4 | Probamos un detector general: ¿ve peces? | YOLO |
| 5 | Le enseñamos a ver peces (*transfer learning*) | YOLO |
| 6 | **En vivo** sobre el video del barco | YOLO + OpenCV |
| 7 | El umbral, otra vez | YOLO |
| 8 | Lo que este modelo todavía no sabe | — |

> ⚡ **Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**. Sin GPU también funciona, pero el entrenamiento de la Parte 5 se hace corto (y el modelo sale peor).

### De dónde salen los datos (todo abierto)
- **Video:** *Tutoriel vidéo campagne halieutique — fonctionnement de la salle de tri du N/O Thalassa*, Ifremer (S. Lesbats, F. Garren, D. Le Roy), **CC BY 4.0**, vía [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Tutoriel_vid%C3%A9o_campagne_halieutique_-_fonctionnement_de_la_salle_de_tri_du_N-O_Thalassa_(Ifremer_00624-73650).webm). Es la sala donde se clasifica y mide la captura en las campañas de evaluación del buque oceanográfico francés *Thalassa*.
- **Fotos para entrenar:** *fish market* de Roboflow 100, **CC BY 4.0**, vía [Hugging Face](https://huggingface.co/datasets/Francesco/fish-market-ggjso) (fotos de peces enteros en mercados, con su recuadro marcado a mano).
- **Modelos de partida:** YOLO11 (Ultralytics) y ResNet18 (torchvision), preentrenados con datasets públicos (COCO e ImageNet).

---
## Parte 0 — Preparar el entorno
Instalamos `ultralytics` (la librería de los modelos YOLO). OpenCV, PyTorch y el resto ya vienen en Colab.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, time, json, urllib.request
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch, torchvision
from IPython.display import display, HTML, Image as ImagenIPy
from ultralytics import YOLO

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12})

# Paleta del curso
UTNAZUL, UTNCELESTE, PESCATEAL, TECHGOLD = '#00467F', '#0099CC', '#059669', '#D4A017'

HAY_GPU = torch.cuda.is_available()
DISPOSITIVO = 0 if HAY_GPU else 'cpu'
print('GPU disponible:', HAY_GPU, '·', torch.cuda.get_device_name(0) if HAY_GPU else 'trabajamos con CPU')
print('OpenCV', cv2.__version__, '· PyTorch', torch.__version__)

---
## Parte 1 — El video real: la sala de clasificación del *Thalassa*

Bajamos una versión liviana del video (640×360 px, 67 MB). Para OpenCV un video es simplemente una **pila de fotos**: 25 por segundo.

In [ ]:
URL_VIDEO = ('https://upload.wikimedia.org/wikipedia/commons/transcoded/f/f3/'
             'Tutoriel_vid%C3%A9o_campagne_halieutique_-_fonctionnement_de_la_salle_de_tri_du_N-O_Thalassa_%28Ifremer_00624-73650%29.webm/'
             'Tutoriel_vid%C3%A9o_campagne_halieutique_-_fonctionnement_de_la_salle_de_tri_du_N-O_Thalassa_%28Ifremer_00624-73650%29.webm.360p.mpeg4.mov')
ARCHIVO_VIDEO = 'thalassa_sala_de_tri.mov'

if not os.path.exists(ARCHIVO_VIDEO):
    pedido = urllib.request.Request(URL_VIDEO, headers={'User-Agent': 'CursoIA-PesquerosEnIA/1.0 (uso educativo)'})
    with urllib.request.urlopen(pedido) as r, open(ARCHIVO_VIDEO, 'wb') as f:
        f.write(r.read())

video = cv2.VideoCapture(ARCHIVO_VIDEO)
cuadros_por_segundo = video.get(cv2.CAP_PROP_FPS)
total_cuadros = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
ancho, alto = int(video.get(cv2.CAP_PROP_FRAME_WIDTH)), int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'{ancho}×{alto} px · {cuadros_por_segundo:.0f} cuadros por segundo · {total_cuadros:,} cuadros · {total_cuadros/cuadros_por_segundo/60:.1f} minutos')

def leer_cuadro(segundo):
    # Devuelve el cuadro del segundo pedido (en BGR, como lo usa OpenCV)
    video.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)
    ok, cuadro = video.read()
    return cuadro

def mostrar(cuadro_bgr, titulo='', ax=None):
    ax = ax or plt.gca()
    ax.imshow(cv2.cvtColor(cuadro_bgr, cv2.COLOR_BGR2RGB)); ax.set_title(titulo); ax.axis('off')

Recorramos el video en ocho escenas. Los subtítulos en francés vienen grabados en el video original.

In [ ]:
escenas = {
    20:  'Especies sensibles: al tanque',
    60:  'Clasificación por especie en la cinta',
    70:  'Fin de cinta: la especie dominante',
    122: 'Cajas listas: «petit» y «gros»',
    152: 'Muestreo: se divide la caja',
    208: 'Medición de talla en el ictiómetro',
    270: 'Muestra de escamas',
    300: 'Eglefino: peso, sexo y madurez',
}
fig, ejes = plt.subplots(2, 4, figsize=(16, 6.4))
for ax, (seg, texto) in zip(ejes.flat, escenas.items()):
    mostrar(leer_cuadro(seg), f'{seg} s · {texto}', ax)
plt.suptitle('La sala de clasificación del N/O Thalassa (Ifremer, CC BY 4.0)', fontsize=14, color=UTNAZUL)
plt.tight_layout(); plt.show()

---
## Parte 2 — ¿Qué es una convolución? La hacemos a mano

Una **convolución** pasa una ventanita de números (el **filtro** o *kernel*, acá de 3×3) por toda la imagen. En cada lugar multiplica los píxeles por los números del filtro y suma. Según los números, el filtro **desenfoca**, **marca bordes verticales**, **marca bordes horizontales** o **realza detalles**.

Lo hacemos sobre el eglefino del segundo 300, el que la pantalla del barco registra con **40,0 cm**.

In [ ]:
cuadro_eglefino = leer_cuadro(300)
gris = cv2.cvtColor(cuadro_eglefino, cv2.COLOR_BGR2GRAY).astype(np.float32)

filtros = {
    'Desenfoque (promedio)':   np.ones((3, 3), np.float32) / 9,
    'Bordes verticales':       np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),
    'Bordes horizontales':     np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], np.float32),
    'Realce de detalles':      np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
}
print('Así se ve un filtro por dentro (bordes verticales):')
print(filtros['Bordes verticales'])

fig, ejes = plt.subplots(2, 3, figsize=(15, 6))
mostrar(cuadro_eglefino, 'Cuadro original (300 s)', ejes[0, 0])
ejes[0, 1].imshow(gris, cmap='gray'); ejes[0, 1].set_title('En escala de grises'); ejes[0, 1].axis('off')
for ax, (nombre, k) in zip(list(ejes.flat)[2:], filtros.items()):
    salida = cv2.filter2D(gris, -1, k)
    if 'Bordes' in nombre:   # los bordes dan valores + y −: miramos su tamaño y estiramos el contraste
        salida = np.clip(np.abs(salida) / np.percentile(np.abs(salida), 99), 0, 1)
    ax.imshow(salida, cmap='gray'); ax.set_title(nombre); ax.axis('off')
plt.suptitle('Cuatro filtros de 3×3 hechos a mano con OpenCV', fontsize=14, color=UTNAZUL)
plt.tight_layout(); plt.show()

> 💡 **La idea clave:** en estos cuatro filtros **los números los elegimos nosotros**. Una red convolucional tiene **millones** de números así y **los aprende sola** a partir de ejemplos. Las primeras capas terminan pareciéndose a estos filtros de bordes; las siguientes combinan bordes en texturas (escamas, aletas) y las últimas en objetos (un pez, una mano, un cuchillo).

---
## Parte 3 — Mirar dentro de una CNN entrenada

Usamos **ResNet18**, una CNN entrenada con más de un millón de fotos de ImageNet (1.000 categorías). Le pasamos el mismo cuadro y espiamos lo que sale de cuatro de sus capas. Cada imagen chica es un **mapa de activación**: dónde se "encendió" un filtro aprendido.

In [ ]:
pesos = torchvision.models.ResNet18_Weights.DEFAULT
resnet = torchvision.models.resnet18(weights=pesos).eval()
preparar = pesos.transforms()

activaciones = {}
for nombre in ['layer1', 'layer2', 'layer3', 'layer4']:
    getattr(resnet, nombre).register_forward_hook(lambda m, e, s, n=nombre: activaciones.__setitem__(n, s.detach()[0]))

rgb = cv2.cvtColor(cuadro_eglefino, cv2.COLOR_BGR2RGB)
entrada = preparar(torch.from_numpy(rgb).permute(2, 0, 1)).unsqueeze(0)
with torch.no_grad():
    probabilidades = resnet(entrada).softmax(1)[0]

capas = {'layer1': 'Capa 1 (bordes)', 'layer2': 'Capa 2 (texturas)', 'layer3': 'Capa 3 (partes)', 'layer4': 'Capa 4 (objetos)'}
fig, ejes = plt.subplots(4, 6, figsize=(15, 8.5))
for fila, (clave, titulo) in enumerate(capas.items()):
    a = activaciones[clave]
    mas_activos = a.mean(dim=(1, 2)).argsort(descending=True)[:6]
    for col, canal in enumerate(mas_activos):
        ejes[fila, col].imshow(a[canal], cmap='magma'); ejes[fila, col].axis('off')
        ejes[fila, col].set_title(f'{titulo} · filtro {canal.item()}' if col == 0 else f'filtro {canal.item()}', fontsize=9)
plt.suptitle('Los 6 filtros más activos en cada capa de ResNet18', fontsize=14, color=UTNAZUL)
plt.tight_layout(); plt.show()
print('Tamaño de cada mapa:', {capas[k]: tuple(v.shape[1:]) for k, v in activaciones.items()})

Fijate cómo los mapas se van **achicando** (de 56×56 a 7×7) y volviendo más **abstractos**: la red pierde detalle de píxel y gana idea de "qué hay".

¿Y qué cree ResNet18 que hay en el cuadro? Miremos sus cinco respuestas más probables:

In [ ]:
categorias = pesos.meta['categories']
mejores = probabilidades.topk(5)
for p, i in zip(mejores.values, mejores.indices):
    print(f'{categorias[i]:<28} {p.item():6.1%}')

Cuando lo probamos, la respuesta más probable fue **«scabbard»** (la vaina de una espada), seguida de abrelatas, cuchilla y martillo. ImageNet tiene algunas categorías de peces (vivos, fotografiados por aficionados), pero **ninguna de "pescado sobre la mesa de muestreo"**. La red ve una forma **alargada, plateada y brillante sobre una superficie de metal** y responde con lo más parecido que conoce. Esto anticipa la lección de la próxima parte.

---
## Parte 4 — Un detector general: ¿ve peces?

**YOLO11** es un detector convolucional que en cada cuadro dibuja recuadros alrededor de los objetos que conoce. La versión estándar se entrenó con **COCO**: 80 categorías de la vida cotidiana. Veamos cuáles son y si "pez" está entre ellas.

In [ ]:
detector_general = YOLO('yolo11n.pt')
clases_coco = list(detector_general.names.values())
print(len(clases_coco), 'clases. Algunas:', ', '.join(clases_coco[:12]), '…')
print('¿Está "fish" entre las clases?', 'fish' in clases_coco)

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, seg in zip(ejes, [60, 152, 270, 300]):
    r = detector_general.predict(leer_cuadro(seg), conf=0.25, device=DISPOSITIVO, verbose=False)[0]
    vistos = [r.names[int(c)] for c in r.boxes.cls]
    mostrar(r.plot(line_width=2), f'{seg} s · ve: {", ".join(sorted(set(vistos))) or "nada"}', ax)
plt.suptitle('El detector general ve personas… pero no ve el pescado', fontsize=14, color=UTNAZUL)
plt.tight_layout(); plt.show()

> 🎯 **Primera lección de planta:** un modelo "que anda" no anda en **tu** problema si nunca vio **tus** datos. Reconoce a las personas porque en COCO hay miles de fotos de personas; el pescado no está entre sus clases, así que, en el mejor de los casos, no lo marca, y en el peor lo confunde con lo más parecido que conoce (mirá qué dice del eglefino del segundo 300).

---
## Parte 5 — Le enseñamos a ver peces (*transfer learning*)

No empezamos de cero: tomamos el mismo YOLO11 (que **ya sabe ver bordes, texturas y formas**) y lo reentrenamos con **fotos de peces con su recuadro marcado**. Usamos 1.202 fotos del dataset *fish market* y juntamos todas las especies en una sola clase: **pez**.

Primero bajamos las fotos (70 MB) y las pasamos al formato que usa YOLO: una carpeta de imágenes y, por cada imagen, un `.txt` con `clase x_centro y_centro ancho alto` (en proporciones de 0 a 1).

In [ ]:
URL_FOTOS = 'https://huggingface.co/datasets/Francesco/fish-market-ggjso/resolve/main/data/validation-00000-of-00001-dc512a8c47d3da47.parquet'
if not os.path.exists('fotos_peces.parquet'):
    urllib.request.urlretrieve(URL_FOTOS, 'fotos_peces.parquet')
fotos = pd.read_parquet('fotos_peces.parquet')
print(len(fotos), 'fotos ·', sum(len(o['bbox']) for o in fotos['objects']), 'peces marcados')

rng = np.random.default_rng(0)
orden = rng.permutation(len(fotos))
corte = int(0.85 * len(orden))
for parte, indices in [('train', orden[:corte]), ('val', orden[corte:])]:
    os.makedirs(f'peces/images/{parte}', exist_ok=True); os.makedirs(f'peces/labels/{parte}', exist_ok=True)
    for i in indices:
        fila = fotos.iloc[i]; W, H = fila['width'], fila['height']; nombre = f"pez_{fila['image_id']}"
        with open(f'peces/images/{parte}/{nombre}.jpg', 'wb') as f:
            f.write(fila['image']['bytes'])
        with open(f'peces/labels/{parte}/{nombre}.txt', 'w') as f:
            for x, y, w, h in fila['objects']['bbox']:
                f.write(f'0 {(x + w/2)/W:.5f} {(y + h/2)/H:.5f} {w/W:.5f} {h/H:.5f}\n')
with open('peces.yaml', 'w') as f:
    f.write(f"path: {os.path.abspath('peces')}\ntrain: images/train\nval: images/val\nnames:\n  0: pez\n")
print(f'Entrenamiento: {corte} fotos · Validación: {len(orden) - corte} fotos (el modelo nunca las ve al aprender)')

In [ ]:
fig, ejes = plt.subplots(1, 6, figsize=(17, 3.2))
for ax, i in zip(ejes, orden[:6]):
    fila = fotos.iloc[i]
    img = cv2.imdecode(np.frombuffer(fila['image']['bytes'], np.uint8), cv2.IMREAD_COLOR)
    for x, y, w, h in fila['objects']['bbox']:
        cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), (0, 200, 0), 4)
    mostrar(img, 'ejemplo marcado a mano', ax)
plt.suptitle('Así son los ejemplos con los que aprende', fontsize=14, color=UTNAZUL)
plt.tight_layout(); plt.show()

Ahora sí, **entrenamos**. Una **época** es una pasada completa por las fotos de entrenamiento. Con GPU hacemos 15 épocas (≈ 3–4 minutos); sin GPU, 2 épocas con imágenes más chicas.

In [ ]:
EPOCAS = 15 if HAY_GPU else 2
TAMANO_IMG = 640 if HAY_GPU else 416

modelo_peces = YOLO('yolo11n.pt')          # partimos del detector general…
inicio = time.time()
modelo_peces.train(data='peces.yaml', epochs=EPOCAS, imgsz=TAMANO_IMG, batch=16, device=DISPOSITIVO,
                   name='peces', exist_ok=True, plots=False, verbose=False)
CARPETA_ENTRENAMIENTO = str(modelo_peces.trainer.save_dir)   # donde quedaron los resultados y los pesos
print(f'\nEntrenamiento terminado en {(time.time() - inicio)/60:.1f} minutos')

In [ ]:
historia = pd.read_csv(f'{CARPETA_ENTRENAMIENTO}/results.csv')
historia.columns = historia.columns.str.strip()
final = historia.iloc[-1]
print(f"Precisión (de lo que marcó como pez, cuánto era pez): {final['metrics/precision(B)']:.0%}")
print(f"Exhaustividad (de los peces que había, cuántos encontró): {final['metrics/recall(B)']:.0%}")
print(f"mAP50 (nota global de detección, 0 a 1): {final['metrics/mAP50(B)']:.2f}")

plt.figure(figsize=(7, 3.2))
plt.plot(historia['epoch'], historia['metrics/mAP50(B)'], marker='o', color=PESCATEAL)
plt.title('Cómo mejora con cada época (fotos de validación)'); plt.xlabel('Época'); plt.ylabel('mAP50'); plt.ylim(0, 1); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

modelo_peces = YOLO(f'{CARPETA_ENTRENAMIENTO}/weights/best.pt')   # nos quedamos con la mejor época

---
## Parte 6 — En vivo sobre el video del barco

Ojo: el modelo **aprendió con fotos de mercado** y ahora lo ponemos frente a la mesa de muestreo de un buque francés que **nunca vio**: otra luz, otras especies, manos y guantes en el medio. Recorremos dos tramos con piezas sobre el ictiómetro y vemos, cuadro por cuadro, qué detecta.

> 👀 **Mientras corre, anotá:** ¿en qué tomas acierta? ¿Qué cosas que **no** son peces marca como pez? ¿Qué le pasa con el calamar?

In [ ]:
TRAMOS = [(198, 245), (265, 310)]   # segundos del video
PASO = 5                            # analizamos 1 de cada 5 cuadros (5 por segundo)
UMBRAL = 0.25                       # confianza mínima para dibujar un recuadro

def anotar(cuadro, modelo, umbral=UMBRAL):
    r = modelo.predict(cuadro, conf=umbral, device=DISPOSITIVO, verbose=False)[0]
    return r.plot(line_width=2), len(r.boxes), (r.boxes.conf.max().item() if len(r.boxes) else 0.0)

pantalla = display(HTML('<i>Arrancando…</i>'), display_id=True)
registro = []                        # (segundo, peces detectados, confianza máxima)
salida = cv2.VideoWriter('thalassa_detecciones.mp4', cv2.VideoWriter_fourcc(*'mp4v'), cuadros_por_segundo / PASO, (ancho, alto))
inicio = time.time()
for desde, hasta in TRAMOS:
    for n in range(int(desde * cuadros_por_segundo), int(hasta * cuadros_por_segundo), PASO):
        video.set(cv2.CAP_PROP_POS_FRAMES, n); ok, cuadro = video.read()
        if not ok: break
        dibujado, cantidad, conf_max = anotar(cuadro, modelo_peces)
        segundo = n / cuadros_por_segundo
        registro.append((segundo, cantidad, conf_max))
        cv2.putText(dibujado, f'{segundo:5.1f} s  peces: {cantidad}', (10, alto - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        salida.write(dibujado)
        ok, jpg = cv2.imencode('.jpg', dibujado)
        pantalla.update(ImagenIPy(data=jpg.tobytes()))
salida.release()
duracion = time.time() - inicio
print(f'{len(registro)} cuadros analizados en {duracion:.0f} s → {len(registro)/duracion:.1f} cuadros por segundo con {"GPU" if HAY_GPU else "CPU"}')

Miremos el resumen de lo que vio a lo largo de los dos tramos.

In [ ]:
reg = pd.DataFrame(registro, columns=['segundo', 'peces', 'confianza_max'])
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(reg['segundo'], reg['confianza_max'], '.', color=UTNCELESTE, label='confianza máxima en el cuadro')
ax.axhline(UMBRAL, color=TECHGOLD, ls='--', label=f'umbral = {UMBRAL}')
ax.set_xlabel('Segundo del video'); ax.set_ylabel('Confianza'); ax.set_ylim(0, 1)
ax.set_title('¿Qué tan seguro estuvo el modelo en cada cuadro?'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Cuadros con al menos un pez detectado: {(reg['peces'] > 0).mean():.0%}")

**Lo que vimos cuando lo probamos** (con 3 épocas; con GPU y 15 épocas los números cambian, pero el patrón se repite):
- ✅ Encuentra varias piezas sobre el ictiómetro y en la mano, con confianza **0,3 a 0,7**: bastante **menos segura** que con las fotos de mercado (donde superaba 0,9).
- ❌ Marca el **calamar** como "pez" (hasta 0,61): es alargado y está sobre el ictiómetro, y nunca le mostramos un calamar como "no pez".
- ❌ Marca como "pez" un **guante azul**, una **mano con un tubo de muestra** y hasta el **ictiómetro vacío**: formas claras y alargadas, parecidas a lo que aprendió.
- ❌ Dibuja recuadros **demasiado grandes**: en las fotos de mercado el pez ocupa casi toda la imagen, y eso también lo aprendió.

Es exactamente lo que vimos en la clase: **un modelo rinde peor cuando cambian la luz, la cámara y el contexto**. En la planta, eso se resuelve con fotos propias.

Y el video anotado completo, para verlo de corrido (en Colab se convierte a un formato que el navegador reproduce):

In [ ]:
from base64 import b64encode
os.system('ffmpeg -y -loglevel error -i thalassa_detecciones.mp4 -vcodec libx264 -pix_fmt yuv420p thalassa_detecciones_web.mp4')
archivo = 'thalassa_detecciones_web.mp4' if os.path.exists('thalassa_detecciones_web.mp4') else 'thalassa_detecciones.mp4'
datos = b64encode(open(archivo, 'rb').read()).decode()
display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{datos}" type="video/mp4"></video>'))

---
## Parte 7 — El umbral, otra vez

Igual que en el simulador **«La cámara en la línea»**: el modelo no dice "es pez" o "no es pez"; da un **puntaje de confianza**, y **nosotros** elegimos desde qué puntaje lo damos por bueno. Mismo cuadro, cuatro umbrales:

In [ ]:
cuadro_prueba = leer_cuadro(277)
fig, ejes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, u in zip(ejes, [0.05, 0.25, 0.50, 0.75]):
    dibujado, cantidad, _ = anotar(cuadro_prueba, modelo_peces, umbral=u)
    mostrar(dibujado, f'umbral {u:.2f} → {cantidad} recuadro(s)', ax)
plt.suptitle('Umbral bajo: aparecen falsos peces · Umbral alto: se escapan los verdaderos', fontsize=13, color=UTNAZUL)
plt.tight_layout(); plt.show()

En nuestra prueba, en este cuadro aparecen **un pez verdadero** (sobre el ictiómetro) y **un falso pez** (la mano con el tubo de muestra). Con umbral bajo se ven los dos; al subirlo, primero desaparece el falso… y si seguimos subiendo, también el verdadero.

> ⚖️ ¿Dónde lo pondrías **si cada pez no detectado es una pieza que no se mide**? ¿Y **si cada falso pez dispara un expulsor** en la línea? La respuesta no es técnica: es una **decisión de la empresa**.

---
## Parte 8 — Lo que este modelo todavía no sabe

En unos minutos pasamos de "no ve peces" a "encuentra la pieza sobre la mesa". Es el poder del *transfer learning*. Pero seamos honestos con lo que **no** hace:

| No sabe… | Porque… | Lo que haría falta en una planta |
|---|---|---|
| Separar peces **amontonados** en una caja | Aprendió con **una pieza por foto** | Fotos de cajas y cintas reales, marcadas pieza por pieza |
| Distinguir un pez de un **guante, una mano, un calamar o el ictiómetro vacío** | Nunca le mostramos nada de eso como "no pez" | Fotos de la línea real, con todo lo que aparece en ella |
| Dibujar el recuadro **justo** | En sus fotos el pez llenaba la imagen | Ejemplos con piezas de todos los tamaños y posiciones |
| Decir la **especie** | Juntamos todas en una clase | Etiquetar especie (merluza, calamar, langostino…) |
| Ver **defectos** (hematomas, parásitos, *gaping*) | Nunca le mostramos uno | Cientos de ejemplos de cada defecto, con luz controlada |
| Medir la **talla en cm** | Un recuadro está en píxeles | Calibrar la cámara con una referencia fija (como el ictiómetro) |
| Seguir andando si **cambia la luz** | Sus datos tienen una sola "luz" | Monitorear la deriva y reentrenar (ver la clase de hoy) |

> 🧭 **Para llevarse:** la CNN aprende a ver **lo que le mostramos**. El trabajo de verdad en una planta no es el modelo: son **los datos propios, bien etiquetados, con la luz bajo control**.

---
### 🧪 Desafíos para seguir
1. Cambiá `TRAMOS` por `[(115, 160)]` (las cajas) y mirá cómo le va con el pescado amontonado. ¿Por qué?
2. Bajá `UMBRAL` a 0.10 en la Parte 6: ¿cuántos cuadros con "pez" ganás y cuántos falsos aparecen?
3. **Medir talla:** en el segundo 300 la pantalla del barco dice **40,0 cm**. Medí el largo del recuadro en píxeles y calculá cuántos píxeles son un centímetro en esa toma. ¿Te sirve el mismo número para la pieza del segundo 270? ¿Qué condición tendría que cumplir la cámara?

### Para explorar más
- [Video original en Wikimedia Commons (Ifremer, CC BY 4.0)](https://commons.wikimedia.org/wiki/File:Tutoriel_vid%C3%A9o_campagne_halieutique_-_fonctionnement_de_la_salle_de_tri_du_N-O_Thalassa_(Ifremer_00624-73650).webm)
- [Dataset *fish market* (Roboflow 100, CC BY 4.0)](https://huggingface.co/datasets/Francesco/fish-market-ggjso)
- [Documentación de Ultralytics YOLO](https://docs.ultralytics.com/) · [Tutoriales de OpenCV en Python](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [Modelos abiertos de NOAA Fisheries (NMFS Open Science) en Hugging Face](https://huggingface.co/NMFS-OSI): detectores de peces y otros organismos marinos
- Simuladores de la clase: [La cámara en la línea](https://pesquerosenia.github.io/curso-ia-produccion-pesquera/interactivos/c7_camara_linea.html) · [El ojo y la IA](https://pesquerosenia.github.io/curso-ia-produccion-pesquera/interactivos/planta_inspector.html)

*Ariel Giamportone | PesquerosEnIA · UTN FRCh · 2026*